# AU-Model Pretraining — Colab H100 Runner

Run cells **top-to-bottom** on a fresh session.  
Training resumes automatically if a checkpoint already exists in Drive — no extra step needed.

## Cell 1: Mount Google Drive & configure output paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── Edit these paths to match your Drive layout ──────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/AU_MODEL'
SHARD_DIR    = f'{DRIVE_ROOT}/data/shards'       # root containing wikipedia/, oscar/, etc.
OUTPUT_DIR   = f'{DRIVE_ROOT}/checkpoints/pretrain'
VAL_SHARD    = f'{SHARD_DIR}/val/shard_00000.bin'  # optional; set to '' to skip validation
WANDB_PROJECT = 'au-model'
RUN_NAME     = None   # None → auto-generated by W&B
# ─────────────────────────────────────────────────────────────────────────────

import os, pathlib
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'Drive mounted.  Shard dir: {SHARD_DIR}  |  Output dir: {OUTPUT_DIR}')

## Cell 2: Install dependencies

In [ ]:
# flash-attn must be built from source (--no-build-isolation) because Colab
# provides its own CUDA headers; wandb + tqdm are lightweight runtime deps.
!pip install flash-attn --no-build-isolation wandb tqdm -q
print('Dependencies installed.')

## Cell 3: Hardware configuration — TF32, SDPA backend, GPU info

In [ ]:
import torch

# Enable TF32 on Ampere / Hopper GPUs (H100 is Hopper)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Prefer FlashAttention-2 via PyTorch SDPA where available
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)

# GPU report
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f'GPU      : {props.name}')
    print(f'VRAM     : {vram_gb:.1f} GB')
    print(f'CUDA     : {torch.version.cuda}')
    print(f'PyTorch  : {torch.__version__}')
    if vram_gb < 70:
        print('[WARNING] VRAM < 70 GB — reduce micro_batch_size to avoid OOM (SC-008 target: ≥70 GB with micro_batch_size=32)')
else:
    print('[WARNING] No GPU detected — training on CPU will be extremely slow.')

## Cell 4: Clone / pull repo & configure Python path

In [ ]:
import subprocess, sys, os

REPO_URL  = 'https://github.com/YOUR_USERNAME/AU_MODEL.git'  # ← update before running
REPO_DIR  = '/content/AU_MODEL'
GIT_BRANCH = 'master'

if os.path.isdir(REPO_DIR):
    print('Repo exists — pulling latest changes ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    print('Cloning repo ...')
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, REPO_URL, REPO_DIR], check=True)

# Make project root importable without installing the package
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f'Working directory: {os.getcwd()}')

## Cell 5: Model sanity check

In [ ]:
# Runs the existing sanity_check script which verifies:
#   • Model instantiates without error
#   • Forward pass returns (logits, loss) with correct shapes
#   • Parameter count matches expected ~749 M
exec(open('model/sanity_check.py').read())

## Cell 6: Start (or resume) pretraining

Resume is **automatic**: if `OUTPUT_DIR` contains a `step_NNNNNN.pt` checkpoint the trainer picks it up and continues from `step+1`.  
Edit the `sys.argv` list below to override any `TrainingConfig` field without touching source code.

In [ ]:
import sys

val_shard_arg = ['--val_shard', VAL_SHARD] if VAL_SHARD else []
run_name_arg  = ['--run_name',  RUN_NAME]  if RUN_NAME  else []

sys.argv = [
    'train',
    '--shard_dir',          SHARD_DIR,
    '--output_dir',         OUTPUT_DIR,
    '--wandb_project',      WANDB_PROJECT,
    # Production schedule (constitution-locked defaults kept unless overridden here)
    '--max_steps',          '100000',
    '--warmup_steps',       '2000',
    '--micro_batch_size',   '32',
    '--grad_accum_steps',   '4',
    '--seq_len',            '4096',
    '--log_interval',       '10',
    '--checkpoint_interval','1000',
    '--val_interval',       '1000',
] + val_shard_arg + run_name_arg

from training.trainer import train, parse_args
cfg = parse_args()
print('Config:', cfg)
train(cfg)

## Cell 7: Display W&B run URL

In [ ]:
try:
    import wandb
    if wandb.run is not None:
        print(f'W&B run URL : {wandb.run.url}')
        print(f'Run name    : {wandb.run.name}')
        print(f'Run ID      : {wandb.run.id}')
        wandb.finish()
    else:
        print('No active W&B run (use_wandb=False or init failed).')
except ImportError:
    print('W&B not installed.')